# Sparse Metric Anchors — reproduce every number in the paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/635jack/sparse-metric-anchors/blob/main/colab/reproduce.ipynb)

Companion notebook to *Sparse Metric Anchors for a Single-View 3D Generative Prior: The
Output Frame Is the Bottleneck* (ISIR, Sorbonne Université, 2026).

**On any runtime, in a few minutes:** it fetches the code (`635jack/sparse-metric-anchors`)
and five reference meshes from the benchmark (`jack635/sparse-metric-anchors-ycb`), then
recomputes **every table and figure of the paper from the published campaign results**,
with its own implementation of the statistics, and re-runs live the one result that
needs no generative model: the Poisson witness.

**On a GPU runtime it also generates.** The last section loads the billion-parameter
backbone (`WaLa-SV-1B`) with a low-memory loader (about 6 GB of RAM) and anchors a
generation live. The first cell installs what each section needs, on Python 3.10 to
3.13, and prints which sections this runtime can run.

Every cell prints the paper's value next to the recomputed one.

In [ ]:
# Installs what each section needs, on Python 3.10 to 3.13, then prints what this runtime can run.
# On Python 3.13 one pip line is not enough: open3d has no PyPI wheel, pymcubes has none either
# and builds from source (about a minute), spconv-cu120 stops at Python 3.11, and setuptools 82
# removed pkg_resources, which pytorch_wavelets imports. Re-running this cell is safe.
import sys, os, subprocess, importlib, platform, json, tempfile, urllib.request
from importlib.metadata import version, PackageNotFoundError

PY = sys.version_info[:2]
LINUX_X86 = sys.platform.startswith("linux") and platform.machine() == "x86_64"
try:
    import torch
    CUDA = torch.version.cuda if torch.cuda.is_available() else None
    GPU = torch.cuda.get_device_name(0) if CUDA else ("Apple MPS" if torch.backends.mps.is_available() else None)
except ImportError:
    torch, CUDA, GPU = None, None, None
print(f"Python {PY[0]}.{PY[1]} on {platform.system()} {platform.machine()} | "
      f"torch {torch.__version__ if torch else 'absent'} | GPU: {GPU or 'none'}")

# pip must never replace the runtime's torch: that build is matched to the runtime's CUDA.
# The pin file gets a name of its own: two kernels sharing one file pinned each other's torch.
with tempfile.NamedTemporaryFile("w", suffix="-keep-torch.txt", delete=False) as f:
    PIN = f.name
    for p in ("torch", "torchvision"):
        try:
            f.write(f"{p}=={version(p)}\n")
        except PackageNotFoundError:
            pass

def pip(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-c", PIN, *pkgs],
                       capture_output=True, text=True)
    if r.returncode:  # no check=True: a failure has to be readable, not a traceback
        tail = (r.stderr or r.stdout).strip().splitlines()[-6:]
        print(f"pip could not install {' '.join(pkgs)[:70]}:\n  " + "\n  ".join(tail))
    return r.returncode == 0

def can_import(mod):
    importlib.invalidate_caches()
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False

# --- every runtime: the statistics and the Poisson witness ---------------------------------
pip("numpy", "scipy", "matplotlib", "huggingface_hub")
if not can_import("open3d"):
    if PY <= (3, 12):
        pip("open3d")
    elif LINUX_X86:
        # PyPI stops at cp312; Open3D's development build ships newer wheels. That release is
        # rebuilt continuously, so its wheel is looked up here rather than written down.
        tag, url = f"cp{PY[0]}{PY[1]}", None
        try:
            rel = json.load(urllib.request.urlopen(
                "https://api.github.com/repos/isl-org/Open3D/releases/tags/main-devel", timeout=60))
            url = next((a["browser_download_url"] for a in rel["assets"]
                        if a["name"].startswith("open3d_cpu-") and f"-{tag}-{tag}-manylinux" in a["name"]
                        and a["name"].endswith("_x86_64.whl")), None)
        except Exception as e:
            print("open3d: GitHub API unreachable:", e)
        if url:
            print("open3d: development build", url.rsplit("/", 1)[-1].replace("%2B", "+"))
            pip(url)
        elif url is None:
            print(f"open3d: no development wheel for {tag} on Linux x86_64")
    else:
        print(f"open3d: no wheel for Python {PY[0]}.{PY[1]} on this platform; use Python 3.12")

# --- GPU runtimes only: the generation stack ----------------------------------------------
if GPU:
    pip("setuptools<82", "boto3", "pymcubes", "pytorch-wavelets", "PyWavelets", "einops",
        "timm", "transformers", "pytorch-lightning", "fasteners", "ftfy", "regex", "backoff")
    if CUDA and not can_import("spconv"):
        major, minor = (int(x) for x in CUDA.split(".")[:2])
        # within one CUDA major version, a build for an older minor version runs fine
        fit = [b for M, m, b in [(11, 8, "cu118"), (12, 1, "cu121"), (12, 4, "cu124"), (12, 6, "cu126")]
               if M == major and m <= minor]
        if fit:
            print(f"spconv: torch is built for CUDA {CUDA}, installing spconv-{fit[-1]}")
            pip(f"spconv-{fit[-1]}")
        else:
            print(f"spconv: no prebuilt wheel for CUDA {CUDA} (they exist for 11.8 and 12.1 to 12.6)")

HAVE_O3D = can_import("open3d")
missing = [name for name, ok in [
    ("a GPU", GPU is not None), ("open3d", HAVE_O3D), ("spconv", can_import("spconv")),
    ("mcubes", can_import("mcubes")), ("pytorch_wavelets", can_import("pytorch_wavelets")),
    ("timm", can_import("timm")), ("transformers", can_import("transformers")),
    ("pytorch_lightning", can_import("pytorch_lightning")), ("boto3", can_import("boto3"))] if not ok]
HAVE_GEN = not missing
print()
print("statistics      : yes")
print("Poisson witness :", "yes" if HAVE_O3D else "no, open3d is missing")
print("generation      :", "yes" if HAVE_GEN else "no, missing " + ", ".join(missing))
print("\nIf this cell changed a package the kernel had already imported, restart the kernel and run all.")

In [ ]:
from pathlib import Path
import subprocess, os, time
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import HfHubHTTPError

# The campaign results and the contact sets live in the code repository itself, so one
# git clone brings them all at once. Only the five reference meshes the Poisson witness
# needs come from the dataset -- five files, fetched one at a time with a back-off on
# the Hub's anonymous rate limit (HTTP 429). A full anonymous download of the 680-file
# dataset trips that limit; do not do it from here.
if not Path("sparse-metric-anchors").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/635jack/sparse-metric-anchors"], check=True)
REPO = Path("sparse-metric-anchors").resolve()
sys.path[:0] = [str(REPO / "tools"), str(REPO)]
RES = REPO / "results"
CONTACTS = REPO / "data" / "contacts"

TEST_OBJECTS = ["002_master_chef_can", "006_mustard_bottle", "011_banana", "025_mug", "035_power_drill"]
MESH = {}
for o in TEST_OBJECTS:
    for attempt in range(6):
        try:
            MESH[o] = Path(hf_hub_download("jack635/sparse-metric-anchors-ycb", f"objects/{o}/mesh.obj", repo_type="dataset"))
            break
        except HfHubHTTPError as e:
            if "429" not in str(e) or attempt == 5: raise
            time.sleep(20 * (attempt + 1))
print("code + results :", REPO); print("results :", sorted(p.name for p in RES.glob("*.json")))
print("meshes :", len(MESH), "from the Hub")

In [ ]:
import json, numpy as np
from math import comb

def sign_test(d):
    # two-sided exact sign test, ties discarded -- the paper's protocol
    d = [x for x in d if x != 0]; n = len(d); k = sum(1 for x in d if x > 0)
    p = min(1.0, 2 * sum(comb(n, i) for i in range(0, min(k, n - k) + 1)) / 2 ** n)
    return k, n, p

def gains(fn, cond, base="baseline", key="f2"):
    d = json.load(open(RES / fn))
    return {(o, s): d[o][f"{cond}_s{s}"][key] - d[o][f"{base}_s{s}"][key]
            for o in d for s in (0, 1, 2)
            if f"{cond}_s{s}" in d[o] and f"{base}_s{s}" in d[o]
            and key in d[o][f"{cond}_s{s}"] and key in d[o][f"{base}_s{s}"]}

def baseline(fn, key="f2"):
    d = json.load(open(RES / fn))
    return {(o, s): d[o][f"baseline_s{s}"][key] for o in d for s in (0, 1, 2) if f"baseline_s{s}" in d[o]}

def show(label, value, paper):
    print(f"  {label:44s} recomputed {value:>10}   paper {paper}")

## Table I — constraint anchoring, 42 objects, frame estimated

Anchors are the eight palpation patches drawn from the camera that rendered the images
(`data/contacts_render`). The oracle arm reuses the unanchored passes of the estimated
arm, so the frame cost is a paired difference of anchored scores. The earlier campaign
(`full42_combined.json`, sites drawn from a misplaced camera) is reported as a replication.

In [ ]:
g = gains("full42_render_combined.json", "palpation8_128"); v = np.array(list(g.values()))
b = np.array(list(baseline("full42_render_combined.json").values()))
k, n, p = sign_test(v)
per_obj = {}
for (o, s), x in g.items(): per_obj.setdefault(o, []).append(x)
show("unanchored baseline, F@2 %", f"{b.mean():.1f}", "52.7")
show("mean gain, frame estimated", f"{v.mean():+.2f}", "+3.17")
show("median / std", f"{np.median(v):+.2f} / {v.std(ddof=1):.2f}", "+1.62 / 8.18")
show("positive cases (ties discarded)", f"{k} / {n}", "78 / 126")
show("sign test, two-sided", f"{p:.1e}", "9.5e-03")
show("objects with positive mean gain", f"{sum(np.mean(x) > 0 for x in per_obj.values())} / {len(per_obj)}", "30 / 42")
# The oracle arm anchors the very same unanchored passes, so the frame cost is paired on the anchored scores.
est = json.load(open(RES / "full42_render_combined.json")); ora = json.load(open(RES / "full42_render_oracle.json"))
keys = sorted(g)
go = {c: ora[c[0]][f"palpation8_128_s{c[1]}"]["f2"] - est[c[0]][f"baseline_s{c[1]}"]["f2"] for c in keys}
cost = np.array([ora[o][f"palpation8_128_s{s}"]["f2"] - est[o][f"palpation8_128_s{s}"]["f2"] for o, s in keys])
kc, nc, pc = sign_test(cost)
show("mean gain, frame given (oracle)", f"{np.mean(list(go.values())):+.2f}", "+7.91")
show("frame cost, paired (oracle - estimated)", f"{cost.mean():+.2f} ± {cost.std(ddof=1)/len(cost)**.5:.2f}, {kc}/{nc}", "+4.74 ± 0.68, 102/126")
show("share of the ceiling realised", f"{v.mean()/np.mean(list(go.values())):.0%}", "40%")
# Replication: the earlier, independent campaign, whose palpation sites came from a misplaced camera.
r = gains("full42_combined.json", "palpation8_128"); ro = gains("full42_oracle.json", "palpation8_128")
d = np.array([r[c] - g[c] for c in keys])
show("replication, earlier draw: estimated / oracle", f"{np.mean(list(r.values())):+.2f} / {np.mean(list(ro.values())):+.2f}", "+3.21 / +7.82")
show("earlier - rerun, paired", f"{d.mean():+.2f} ± {d.std(ddof=1)/len(d)**.5:.2f}", "+0.04 ± 0.70 (the two campaigns agree)")

In [ ]:
from notebook_viz import frame_bottleneck
frame_bottleneck(list(gains("full42_render_combined.json", "palpation8_128").values()),
                 list(gains("full42_render_oracle.json", "palpation8_128").values()))

## Table II — the two anchoring routes, and their combination

In [ ]:
mono = json.load(open(RES / "full42_combined.json")); dm4 = json.load(open(RES / "dm4_resultats.json"))
rows = []
for o in mono:
    if o not in dm4: continue
    for s in (0, 1, 2):
        try:
            rows.append((mono[o][f"baseline_s{s}"]["f2"], mono[o][f"palpation8_128_s{s}"]["f2"],
                         dm4[o][f"baseline_s{s}"]["f2"], dm4[o][f"guide_s{s}"]["f2"]))
        except KeyError: pass
rb, rt, db, dt = np.array(rows).T
show("image only", f"{rb.mean():.1f}", "53.1")
show("  + contacts as constraint", f"{rt.mean():.1f}", "56.3")
show("depth reprojected to 4 views", f"{db.mean():.1f}", "64.3")
show("  + contacts as constraint", f"{dt.mean():.1f}", "64.7")
show("per-case oracle choice of route", f"{np.maximum(rt, dt).mean():.1f}", "72.2")
gd = db - rb
show("reprojection gain, paired mean", f"{gd.mean():+.2f}", "+11.22")
show("cases improved / lost", f"{(gd > 0).sum()} / {(gd < 0).sum()}", "87 / 39")
show("mean gain where improved / where lost", f"{gd[gd > 0].mean():+.1f} / {gd[gd < 0].mean():+.1f}", "+26.1 / -22.1")
k, n, p = sign_test(dt - db)
show("contacts on top of anchored frame", f"{(dt - db).mean():+.2f}, {k}/{n}, p={p:.3f}", "+0.38, 75/126, p=0.040")
show("corr(reprojection gain, contact gain)", f"{np.corrcoef(gd, rt - rb)[0, 1]:+.3f}", "+0.056")

In [ ]:
from notebook_viz import route_bimodality
route_bimodality(gd)

## Figure 2a — seven controlled manipulations of the anchor set

In [ ]:
def paired(a, b):
    keys = sorted(set(a) & set(b)); return [a[k] - b[k] for k in keys]
dh = {c: gains("dh116_regime.json", c) for c in ["dh8", "dh8n", "dh8n_bruit", "dh4"]}
st = {c: gains("strategies.json", c) for c in ["fb", "rl"]}
comps = [
  ("8 patches instead of 4",       paired(gains("palpation_combined.json", "palpation_128"), gains("sites_4.json", "palpation4_128")), "+2.22, 12/15, p=0.035"),
  ("32 patches instead of 8",      paired(gains("sites_32.json", "palpation32_128"), gains("palpation_combined.json", "palpation_128")), "-0.24, 5/15, p=0.302"),
  ("512 anchors instead of 32",    paired(gains("guidance_campaign.json", "occluded_512"), gains("guidance_campaign.json", "occluded_32")), "-0.16, 8/15, p=1.000"),
  ("surface normals added",        paired(dh["dh8n"], dh["dh8"]), "+0.93, 10/15, p=0.302"),
  ("anchors off the hidden face",  paired(st["rl"], st["fb"]), "+0.82, 11/15, p=0.118"),
  ("placement noise 3 mm / 15 deg", paired(dh["dh8n_bruit"], dh["dh8n"]), "-2.40, 3/15, p=0.035"),
  ("4 zones instead of 8",         paired(dh["dh4"], dh["dh8"]), "-2.55, 3/15, p=0.035"),
]
for lab, v, paper in comps:
    k, n, p = sign_test(v); show(lab, f"{np.mean(v):+.2f}, {k}/{n}, p={p:.3f}", paper)

In [ ]:
from notebook_viz import manipulations
manipulations(comps, sign_test)

## Figure 2b — contact visibility does not predict the gain

Visibility is decided here from the camera that rendered the images. The campaign placed
that camera in the meshes' own Y-up frame, 68° away, and stored what it gave; the last
line reproduces that value (paper, Limitations; `tools/visibility_render_camera.py`).

In [ ]:
import matplotlib.pyplot as plt
# Blender's OBJ importer turned each mesh (x, y, z) -> (x, -z, y) before the camera at
# (2.2, -2.2, 1.8) rendered it; in the mesh frame that camera is here, scaled 2.0 / 1.3.
BLENDER_IMPORT = np.array([[1., 0., 0.], [0., 0., -1.], [0., 1., 0.]])
CAM_RENDER = BLENDER_IMPORT.T @ np.array([2.2, -2.2, 1.8]) * (2.0 / 1.3)
d = json.load(open(RES / "strategies.json")); grasps = np.load(CONTACTS / "strategies.npz", allow_pickle=True)
STRAT = {"fb": "front_back", "lr": "left_right", "rl": "right_left"}
occ, stored, gain = [], [], []
for o in d:
    for c, strat in STRAT.items():
        pos, nrm = grasps[f"{o}|{strat}|pos"], grasps[f"{o}|{strat}|nrm"]
        nrm = nrm / np.linalg.norm(nrm, axis=1, keepdims=True)
        to_cam = (CAM_RENDER - pos) / np.linalg.norm(CAM_RENDER - pos, axis=1, keepdims=True)
        for s in (0, 1, 2):
            occ.append(float(((nrm * to_cam).sum(1) < 0).mean()))       # the campaign's normal test
            stored.append(d[o][f"{c}_s{s}"]["part_occultee"])
            gain.append(d[o][f"{c}_s{s}"]["f2"] - d[o][f"baseline_s{s}"]["f2"])
occ, stored, gain = np.array(occ), np.array(stored), np.array(gain)
by_strategy = lambda v: [v.reshape(len(d), 3, 3)[:, k].mean() for k in range(3)]
show("hidden anchors, fb / lr / rl", " / ".join(f"{100 * x:.0f}%" for x in by_strategy(occ)), "60% / 87% / 30%")
show("gain, fb / lr / rl", " / ".join(f"{x:+.2f}" for x in by_strategy(gain)), "+1.73 / +1.37 / +2.55")
show("correlation(hidden fraction, gain), 45 cases", f"{np.corrcoef(occ, gain)[0, 1]:+.3f}", "-0.001")
show("cases with no anchor hidden from the camera", f"{(occ == 0).sum()}", "0")
show("same correlation, campaign's misplaced camera", f"{np.corrcoef(stored, gain)[0, 1]:+.3f}", "-0.145 (the error)")
a, b = np.polyfit(occ, gain, 1)
plt.figure(figsize=(4.5, 4)); plt.scatter(occ, gain, s=14, alpha=.7)
x = np.linspace(occ.min(), occ.max(), 2); plt.plot(x, a * x + b, "r-")
plt.axhline(0, c="k", lw=.5); plt.xlabel("fraction of anchors hidden from the camera"); plt.ylabel("gain, F@2 %")
plt.title(f"r = {np.corrcoef(occ, gain)[0, 1]:+.3f}"); plt.show()

## Table III — the 2 × 2 anchor-set design, and the depth-camera control

Oracle frame, 42 objects × 3 seeds, one shared unanchored pass in every cell, 128
anchors each: eight palpation patches or uniform coverage, on the hidden or the visible
face. Contrasts are taken on the anchored scores, so the unanchored score cancels. The
visible-face, uniform cell is what a depth camera delivers.

In [ ]:
cells = {"patches_hidden": ("full42_render_oracle.json", "palpation8_128"),
         "patches_visible": ("full42_render_oracle_vispatch.json", "palpation8visible_128"),
         "uniform_hidden": ("full42_render_oracle_occluded.json", "occluded_128"),
         "uniform_visible": ("full42_render_oracle_visible.json", "visible_128")}
est = json.load(open(RES / "full42_render_combined.json"))
keys = [(o, s) for o in sorted(est) for s in (0, 1, 2)]
base = {c: est[c[0]][f"baseline_s{c[1]}"]["f2"] for c in keys}
anch = {name: {c: json.load(open(RES / f))[c[0]][f"{key}_s{c[1]}"]["f2"] for c in keys} for name, (f, key) in cells.items()}
paper = {"patches_hidden": "+7.91", "patches_visible": "+7.31", "uniform_hidden": "+18.24", "uniform_visible": "+15.57"}
for name in cells:
    show(f"gain, {name}", f"{np.mean([anch[name][c] - base[c] for c in keys]):+.2f}", paper[name])
def contrast(a, b, label, paper_value):
    d = np.array([anch[a][c] - anch[b][c] for c in keys]); k, n, p = sign_test(d)
    show(label, f"{d.mean():+.2f} ± {d.std(ddof=1)/len(d)**.5:.2f}, {k}/{n}, p={p:.1e}", paper_value)
contrast("uniform_hidden", "patches_hidden", "spread effect, hidden face", "+10.33 ± 1.05, 115/125")
contrast("uniform_visible", "patches_visible", "spread effect, visible face", "+8.26 ± 0.94, 104/124")
contrast("patches_hidden", "patches_visible", "face effect, patches (hidden - visible)", "+0.60 ± 0.60, p = 0.47")
contrast("uniform_hidden", "uniform_visible", "face effect, uniform (hidden - visible)", "+2.67 ± 0.48, p = 9.2e-07")
contrast("uniform_visible", "patches_hidden", "depth-camera control minus tactile patches", "+7.66 ± 0.96, 103/125")
# Where the gain lands: recall of the side the rendering camera cannot see, from the archived per-case file.
rows = json.load(open(RES / "hidden_side_recall.json"))["rows"]
for arm, paper_value in (("uniform", "+6.71 ± 0.97"), ("patches", "+1.17 ± 0.82")):
    d = np.array([r[f"{arm}_hidden_hidden"] - r[f"{arm}_visible_hidden"] for r in rows])
    show(f"hidden-side recall, hidden-face minus visible-face anchors ({arm})", f"{d.mean():+.2f} ± {d.std(ddof=1)/len(d)**.5:.2f}", paper_value)

## Section III — the output pose is resampled with the noise

In [ ]:
pd_ = json.load(open(RES / "pose_determinism.json")); a = pd_["agrege"]
show("asymmetric objects: n, median inter-seed rotation", f"{a['asymetriques']['n']}, {a['asymetriques']['ecart_median_deg']:.1f} deg", "16, 131.8 deg in this run; 136.5 ± 3.7 over four runs")
show("symmetric objects:  n, median inter-seed rotation", f"{a['symetriques']['n']}, {a['symetriques']['ecart_median_deg']:.1f} deg", "26, 134.4 deg in this run; 135.7 ± 3.7 over four runs")
show("stable below 30 deg: asym / sym", f"{a['asymetriques']['stables_sous_30']} / {a['symetriques']['stables_sous_30']}", "1 / 0")
show("corr(asymmetry, stability)", f"{a['correlation_asymetrie_stabilite']:+.3f}", "-0.124 in this run; -0.14 ± 0.06 over four runs")

In [ ]:
from notebook_viz import pose_spread
pose_spread([x for o in pd_["objets"].values() for x in o["ecarts_deg"]])

## Section VI-G — run-to-run variance: frame re-estimated vs frozen

In [ ]:
for fn, lab in [("plancher.json", "frame re-estimated"), ("plancher_gel.json", "frame frozen")]:
    r = json.load(open(RES / fn))
    t = np.array([x["temoin"] for x in r]); g = np.array([x["guide"] for x in r]); dd = g - t
    show(f"{lab}: sd unanchored / anchored / paired diff", f"{t.std(ddof=1):.2f} / {g.std(ddof=1):.2f} / {dd.std(ddof=1):.2f}",
         "0.49 / 2.29 / 2.50" if "gel" not in fn else "0.68 / 0.70 / 1.29")
sa = np.std([x["guide"] for x in json.load(open(RES / "plancher.json"))], ddof=1)
sb = np.std([x["guide"] for x in json.load(open(RES / "plancher_gel.json"))], ddof=1)
show("share of anchored variance due to frame re-estimation", f"{100 * (sa**2 - sb**2) / sa**2:.0f} %", "91 % (banana; frozen-frame sd equals unanchored sd on 3 more objects)")
# How much of the frozen-frame floor is the scorer: the same saved meshes, re-scored four times each.
sn = json.load(open(RES / "scoring_noise.json"))
show("scorer alone, sd by object (anchored)", " / ".join(f"{r['guide']['scorer_sd']:.2f}" for o, r in sn.items() if not o.startswith("_")), "0.46 / 2.23 / 0.33 / 1.76")
p = sn["_pooled"]
show("scorer alone, pooled sd: unanchored / anchored", f"{p['temoin']['scorer_sd']:.2f} / {p['guide']['scorer_sd']:.2f}", "1.03 / 1.45 (whole floor 1.72 / 1.60)")

## Table IV — the Poisson witness, run live

Six oriented contacts reconstructed by screened Poisson, scored with the paper's own
alignment and metric. This one is *recomputed*, not re-read: its value moves by a few
tenths between runs because both Poisson and the alignment sample points at random
(re-scoring the very same mesh moves F@2 by 0.3 to 2.2 points depending on the object;
see the variance section above).

In [ ]:
if not HAVE_O3D:
    print("skipped: needs open3d (see the first cell)")
else:
    import open3d as o3d
    from eval_fusion import load_and_normalize_mesh, align_prediction_to_gt, compute_chamfer_and_fscore
    from poisson_temoin import poisson, OBJECTS, STRATEGIES
    npz = np.load(CONTACTS / "strategies.npz", allow_pickle=True)
    out, depth = {"6": [], "18": []}, {}
    for obj in OBJECTS:
        gt, _ = load_and_normalize_mesh(MESH[obj])
        for lab, strats in [("6", ["front_back"]), ("18", STRATEGIES)]:
            pos = np.concatenate([npz[f"{obj}|{s}|pos"] for s in strats]); nrm = np.concatenate([npz[f"{obj}|{s}|nrm"] for s in strats])
            best, depth[obj, lab] = max(((compute_chamfer_and_fscore(gt, align_prediction_to_gt(gt, m))["f_scores"]["2.0%"]["f_score"], dp)
                                         for dp, m in ((dp, poisson(pos, nrm, dp)) for dp in (3, 4, 5, 6)) if m is not None),
                                        default=(float("nan"), None))
            out[lab].append(best)
    show("Poisson, 6 oriented contacts, mean F@2 %", f"{np.nanmean(out['6']):.2f}", "19.90 ± 0.36 over five runs")
    show("Poisson, 18 oriented contacts", f"{np.nanmean(out['18']):.2f}", "20.44 ± 0.18 over five runs")
    show("image only, same objects (from Table I data)", f"{np.mean([baseline('full42_combined.json')[(o, s)] for o in OBJECTS for s in (0,1,2)]):.2f}", "56.52")

In [ ]:
if not HAVE_O3D:
    print("skipped: needs open3d (see the first cell)")
else:
    from notebook_viz import poisson_figure
    # The object whose 6-contact score sits closest to the mean, not the most flattering one.
    k = int(np.nanargmin(np.abs(np.array(out["6"]) - np.nanmean(out["6"]))))
    SHOW = OBJECTS[k]
    gt_show, _ = load_and_normalize_mesh(MESH[SHOW])
    poisson_figure(gt_show, {lab: (np.concatenate([npz[f"{SHOW}|{s}|pos"] for s in strats]),
                                   np.concatenate([npz[f"{SHOW}|{s}|nrm"] for s in strats]),
                                   depth[SHOW, lab], out[lab][k])
                             for lab, strats in [("6", ["front_back"]), ("18", STRATEGIES)]}, name=SHOW)

## Generate, for real

Everything above recomputes statistics. This section runs the model: it anchors a
generation with tactile contacts and scores the result against the reference mesh, so
you can see the constraint act rather than read about it.

**Switch the runtime to a GPU** (Runtime → Change runtime type → T4). Cost: about
18 GB of checkpoint to download once, then roughly 6 GB of RAM and a minute per
sample. The 6 GB figure is why this fits: `tools/load_slim.py` memory-maps the
checkpoint and assigns the tensors instead of copying them, where the stock loader
reads the file twice and peaks at 12.7 GB — above a free runtime's limit. The two
loaders were checked tensor by tensor: **1399 of 1399 bit-identical**.

The last two cells draw what the scores measure: both shapes against the reference,
from the input camera and from the opposite side, coloured by error, then in a
view you can rotate.

In [ ]:
if not HAVE_GEN:
    print("skipped: generation needs what the first cell lists as missing")
else:
    # Downloads ADSKAILab/WaLa-SV-1B on first call (~18 GB). Peak RAM ~6 GB.
    import resource, time
    from load_slim import load_model_slim
    t0 = time.time()
    model, net, device = load_model_slim(timesteps=20)
    rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss      # kilobytes on Linux, bytes on macOS
    print(f"loaded in {time.time()-t0:.0f} s on {device}, "
          f"peak RSS {rss / (1e9 if sys.platform == 'darwin' else 1e6):.1f} GB")

In [ ]:
if not HAVE_GEN:
    print("skipped: generation needs what the first cell lists as missing")
else:
    # One object, one seed: unanchored, then anchored on 8 tactile contacts.
    import numpy as np, torch
    from pathlib import Path
    from huggingface_hub import hf_hub_download
    from guided_sampling import image_condition, make_sampler
    from analyze_occlusion import split_visible_occluded
    from eval_fusion import load_and_normalize_mesh
    from run_guidance_campaign import score, oracle_frame

    # The power drill is the most favourable object in the whole study -- it carries a
    # large share of several totals. Try "011_banana" for a typical case, or "025_mug",
    # where anchoring does nothing. Picking the drill and stopping there is exactly the
    # reading error the paper's statistics exist to prevent.
    OBJ, SEED = "035_power_drill", 0
    img = Path(hf_hub_download("jack635/sparse-metric-anchors-ycb", f"objects/{OBJ}/image.png", repo_type="dataset"))
    gt, _ = load_and_normalize_mesh(MESH[OBJ] if OBJ in MESH else Path(hf_hub_download(
        "jack635/sparse-metric-anchors-ycb", f"objects/{OBJ}/mesh.obj", repo_type="dataset")))
    gt_pts, vis = split_visible_occluded(gt)
    bb = gt.get_axis_aligned_bounding_box()
    thr = 0.02 * float(np.linalg.norm(bb.get_max_bound() - bb.get_min_bound()))

    cond = image_condition(model, net, img, device)
    sample = make_sampler(model, net, cond, OBJ, 1.5, device)

    ref_obj, _ = sample(0.0, SEED, torch.zeros(1, 3, device=device), Path("out/unanchored"))
    base = score(gt, gt_pts, vis, thr, ref_obj)
    print(f"unanchored          F@2 = {base['f2']:.2f}")

    # Oracle placement: an upper bound, not a system. See the paper's discipline note.
    O = oracle_frame(gt, ref_obj)
    pts = torch.from_numpy((O[:3, :3] @ torch.load(
        CONTACTS / OBJ / "palpation8_128.pt", weights_only=True)[0].numpy().T).T + O[:3, 3]
        ).float().to(device)
    pred_obj, trace = sample(0.05, SEED, pts, Path("out/anchored"))
    g = score(gt, gt_pts, vis, thr, pred_obj)
    print(f"anchored, 8 patches F@2 = {g['f2']:.2f}   ({g['f2']-base['f2']:+.2f})")
    print(f"  |SDF| at the contacts: {trace[0][1]:.4f} -> {trace[-1][1]:.4f}   (the constraint being satisfied)")
    print(f"The paper's mean over 42 objects x 3 seeds is +3.21; a single case says nothing on its own")
    print(f"(run-to-run spread of a paired difference is 2.50 -- see the variance section above).")

In [ ]:
if not HAVE_GEN:
    print("skipped: generation needs what the first cell lists as missing")
else:
    # Both predictions aligned onto the reference the way `score` aligns them, then drawn.
    from eval_fusion import align_prediction_to_gt, compute_chamfer_and_fscore
    from notebook_viz import generation_figure, interactive
    aligned = {k: align_prediction_to_gt(gt, load_and_normalize_mesh(p)[0])
               for k, p in [("unanchored", ref_obj), ("anchored", pred_obj)]}
    drawn = {k: {"f2": compute_chamfer_and_fscore(gt, m)["f_scores"]["2.0%"]["f_score"]} for k, m in aligned.items()}
    print("F@2 of the aligned meshes drawn below:", {k: round(float(v["f2"]), 1) for k, v in drawn.items()})
    print("(the alignment samples points at random: these differ from the scores above, see the note below)")
    contacts = torch.load(CONTACTS / OBJ / "palpation8_128.pt", weights_only=True)[0].numpy()
    generation_figure(img, gt, aligned, contacts, drawn, trace, name=OBJ)
    interactive(gt, aligned, contacts)

One case cannot support a claim here, and a figure makes that easy to forget. Re-scoring
the very same two power-drill meshes eight times, without generating anything again,
moves F@2 with a standard deviation of about 2 (65.8 ± 2.1 unanchored, 78.1 ± 1.5
anchored, paired difference +12.3 ± 3.4), because the alignment samples points at
random. Generating again adds the run-to-run spread of the variance section. Every claim
in the paper rests on 15 to 126 paired cases and a sign test, which is the whole point of
that section.